In [1]:
suppressPackageStartupMessages({
  library(data.table)
  library(DESeq2)
  library(ggplot2)
  library(ggpubr)
  library(ggrepel)
  library(grid)
  library(parallel)
  library(qvalue)
  library(rstatix)
  library(reshape2)
  library(stringr)
  library(tidyr)
  library(tibble)
})

# Both - Flu Response

In [2]:
#' Volcano Plot by Cell Type
#'
#' Generates a volcano plot for DESeq2 results by cell type.
#'
#' @param res_file Path to the DESeq2 results CSV file.
#' @param file_save_name Name for the output PDF (without extension).
#' @export
#' @details The input CSV file (`input_deg_res`) must contain the following columns:
#' - `Qvalue`: Numeric values representing the adjusted p-values.
#' - `log2FoldChange`: Numeric values representing the log2 fold change.
#' - `gene`: Character values representing gene names.
#' - `pvalue`: Numeric values representing the raw p-values.
plot_volcano_deg <- function(input_deg_res,
                             title,
                             subtitle,
                             file_save_name = "volcano_plot") {
  if (!file.exists(input_deg_res)) {
    stop("Error: The file does not exist. Please check the file path.")
  }
  res <- tryCatch(
    data.table::fread(input_deg_res),
    error = function(e) {
      stop("Error: Unable to read the file. Please ensure it is a valid CSV file.")
    }
  )
  # Remove rows with NA values in Qvalue or log2FoldChange
  res <- res[!is.na(Qvalue) & !is.na(log2FoldChange), ]
  

  # height <- ceiling(length(unique(res$celltype)) / 4) * 6
  png(paste0("../../../data/rna/pseudobulk/results/deg_volcanos/", file_save_name, "_deg_volcano.png"), width = 20, height = 20, units = "in", res = 300)
  print(
    ggplot2::ggplot(
      res,
      ggplot2::aes(
        x = log2FoldChange,
        y = -log10(pvalue),
        color = ifelse(Qvalue < 0.1 & log2FoldChange > 0.5, "Upregulated in Responders",
          ifelse(Qvalue < 0.1 & log2FoldChange < -0.5, "Upregulated in Non Responders",
            "Not Significant"
          )
        ),
        label = gene
      )
    ) +
      ggplot2::geom_point(size = 1.5) +
      ggrepel::geom_text_repel(
        data = subset(res, Qvalue < 0.1),
        force = 1,
        max.overlaps = 20, 
        show.legend = FALSE
      ) +
      ggplot2::geom_vline(xintercept = c(-0.5, 0.5), linetype = "dashed", color = "grey50", linewidth = 0.5) +
      ggplot2::geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "grey50", linewidth = 0.5) +
      ggplot2::facet_wrap(~celltype, ncol = 4, scales = "free") +
      ggplot2::theme_minimal() +
      ggplot2::labs(
        title = title,
        subtitle = subtitle,
        x = expression(paste("Log"[2], " Fold Change")),
        y = expression(paste("-Log"[10], " (p-value)"))
      ) +
      ggplot2::theme(
        plot.title = ggplot2::element_text(hjust = 0.5, face = "bold", size = 18, margin = ggplot2::margin(b = 10)),
        axis.title = ggplot2::element_text(size = 16, face = "bold"),
        axis.text = ggplot2::element_text(size = 12),
        legend.position = "bottom",
        legend.title = ggplot2::element_blank(),
        legend.text = ggplot2::element_text(size = 12),
        strip.text = ggplot2::element_text(size = 14, hjust = 0.05),
        panel.grid.major = ggplot2::element_line(color = "grey90", linewidth = 0.2),
        panel.grid.minor = ggplot2::element_blank(),
        panel.background = ggplot2::element_rect(fill = "white", color = NA),
        panel.border = ggplot2::element_rect(colour = "black", fill = NA, linewidth = 0.8),
        panel.spacing = grid::unit(0.5, "cm")
      ) +
      ggplot2::scale_color_manual(
        name = NULL,
        values = c(
          "Upregulated in Responders" = "#00b2a8",
          "Upregulated in Non Responders" = "#d81463",
          "Not Significant" = "gray70"
        )
      ) +
      ggplot2::guides(color = ggplot2::guide_legend(override.aes = list(size = 5)))
  )
  dev.off()
}

In [3]:
# List all files in the input folder containing both "bmmc" and "flu"
input_folder <- "../../../data/rna/pseudobulk/results/deseq2_results"
files <- list.files(input_folder, pattern = "results_bmmc.*flu", full.names = TRUE, ignore.case = TRUE)

# Loop through each file and run plot_volcano_deg
for (file in files) {
  visit_detail <- sub(".*flu-([^.]+)\\.csv$", "\\1", basename(file))
  plot_volcano_deg(
    input_deg_res = file,
    title = paste0(
      "BMMC Flu Response Volcano Plot at ", visit_detail, " - (adjusted for sex, plasma cells removed)"
    ),
    subtitle = paste(
      'This plot compares flu response (adjusted for sex) across bone marrow cell types.\nGenes are labeled as "Upregulated in Responders" if Qvalue < 0.1 and log2FoldChange > 0.5,',
      'and "Upregulated in NonResponders" if Qvalue < 0.1 and log2FoldChange < -0.5.'
    ),
    file_save_name = sub('^deseq2_results_', 'l3_', tools::file_path_sans_ext(basename(file)))
  )
}

In [4]:
# List all files in the input folder containing both "pbmc" and "flu"
input_folder <- "../../../data/rna/pseudobulk/results/deseq2_results"
files <- list.files(input_folder, pattern = "results_pbmc.*flu", full.names = TRUE, ignore.case = TRUE)

# Loop through each file and run plot_volcano_deg
for (file in files) {
  visit_detail <- sub(".*flu-([^.]+)\\.csv$", "\\1", basename(file))
  plot_volcano_deg(
    input_deg_res = file,
    title = paste0(
      "PBMC Flu Response Volcano Plot at ", visit_detail
    ),
    subtitle = paste(
      'This plot compares flu response (adjusted for sex) across PBMCs.\nGenes are labeled as "Upregulated in Responders" if Qvalue < 0.1 and log2FoldChange > 0.5,',
      'and "Upregulated in NonResponders" if Qvalue < 0.1 and log2FoldChange < -0.5.'
    ),
    file_save_name = sub('^deseq2_results_', 'l3_', tools::file_path_sans_ext(basename(file)))
  )
}

Warning message:
“ggrepel: 15 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 24 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 19 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 18 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 133 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 8 unlabeled data points (too many overlaps). Consider increasing max.overlaps”


# BMMC - Longitudinal

In [5]:
plot_volcano_by_timepoint <- function(input_deg_res,
                                      title,
                                      subtitle,
                                      timepoint_1,
                                      timepoint_2,
                                      tp_color_map = c(
                                          "PreTx" = "#29216e",
                                          "EI" = "#006dc6",
                                          "ASCT90d" = "#19c2f0",
                                          "ASCT1y" = "#2a7a4c",
                                          "ASCT2y" = "#4FBC3B",
                                          "Healthy" = "#b52267"
                                        ),
                                      file_save_name = "volcano_plot") {
  if (!file.exists(input_deg_res)) {
    stop("Error: The file does not exist. Please check the file path.")
  }

  res <- tryCatch(
    data.table::fread(input_deg_res),
    error = function(e) {
      stop("Error: Unable to read the file. Please ensure it is a valid CSV file.")
    }
  )

  res <- res[!is.na(Qvalue) & !is.na(log2FoldChange), ]

  # Ensure earlier timepoint is always on the left side (negative log2FC)
  tp_order <- c(
    "PreTx",
    "EI",
    "ASCT90d",
    "ASCT1y",
    "ASCT2y"
  )

  # swap removed to preserve label-> LFC semantics (see commit)

  # Assign significance labels
  res$significance <- "Not Significant"
  res$significance[res$Qvalue < 0.1 & res$log2FoldChange > 0.5] <- paste("Upregulated in", timepoint_1)
  res$significance[res$Qvalue < 0.1 & res$log2FoldChange < -0.5] <- paste("Upregulated in", timepoint_2)

  res$significance <- factor(
    res$significance,
    levels = c(
      paste("Upregulated in", timepoint_2),
      paste("Upregulated in", timepoint_1),
      "Not Significant"
    )
  )
  res$significance <- droplevels(res$significance)


  # Safe color lookup with fallbacks if a timepoint is not present in tp_color_map
  color1 <- if (timepoint_1 %in% names(tp_color_map)) tp_color_map[timepoint_1] else "#000000"
  color2 <- if (timepoint_2 %in% names(tp_color_map)) tp_color_map[timepoint_2] else "#888888"
  manual_colors <- setNames(
    c(as.character(color1), as.character(color2), "gray70"),
    c(paste("Upregulated in", timepoint_1), paste("Upregulated in", timepoint_2), "Not Significant")
  )

  # Sanity check: report fraction positive LFC (non-intrusive)
  frac_pos <- mean(res$log2FoldChange > 0, na.rm = TRUE)
  message(sprintf("[volcano sanity] fraction log2FoldChange > 0 = %.3f (timepoint_1=%s)", frac_pos, timepoint_1))
  # Save to PNG
  png(paste0("../../../data/rna/pseudobulk/results/deg_volcanos/", file_save_name, "_deg_volcano.png"), width = 20, height = 28, units = "in", res = 300)
  print(
    ggplot2::ggplot(
      res,
      ggplot2::aes(
        x = log2FoldChange,
        y = -log10(pvalue),
        color = significance,
        label = gene
      )
    ) +
      ggplot2::geom_point(size = 1) +
      ggrepel::geom_text_repel(
        data = subset(res, Qvalue < 0.1),
        force = 1,
        max.overlaps = 20,
        show.legend = FALSE
      ) +
      ggplot2::geom_vline(xintercept = c(-0.5, 0.5), linetype = "dashed", color = "grey50", linewidth = 0.5) +
      ggplot2::geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "grey50", linewidth = 0.5) +
      ggplot2::facet_wrap(~celltype, ncol = 4, scales = "free") +
      ggplot2::theme_minimal() +
      ggplot2::labs(
        title = title,
        subtitle = subtitle,
        x = expression(paste("Log"[2], " Fold Change")),
        y = expression(paste("-Log"[10], " (p-value)"))
      ) +
      ggplot2::theme(
        plot.title = ggplot2::element_text(hjust = 0.5, face = "bold", size = 18, margin = ggplot2::margin(b = 10)),
        axis.title = ggplot2::element_text(size = 16, face = "bold"),
        axis.text = ggplot2::element_text(size = 12),
        legend.position = "bottom",
        legend.title = ggplot2::element_blank(),
        legend.text = ggplot2::element_text(size = 12),
        strip.text = ggplot2::element_text(size = 14, hjust = 0.05),
        panel.grid.major = ggplot2::element_line(color = "grey90", linewidth = 0.2),
        panel.grid.minor = ggplot2::element_blank(),
        panel.background = ggplot2::element_rect(fill = "white", color = NA),
        panel.border = ggplot2::element_rect(colour = "black", fill = NA, linewidth = 0.8),
        panel.spacing = grid::unit(0.5, "cm")
      ) +
      ggplot2::scale_color_manual(values = manual_colors) +
      ggplot2::guides(color = ggplot2::guide_legend(override.aes = list(size = 5)))
  )
  dev.off()
}

## For the BMMC Comparisons

In [6]:
input_folder <- "../../../data/rna/pseudobulk/results/deseq2_results/"
files <- list.files(input_folder, pattern = "deseq2.*bmmc_.*vs.*\\.csv$", full.names = TRUE, ignore.case = TRUE)

for (file in files) {
  base <- sub("^deseq2_results_", "l3_", tools::file_path_sans_ext(basename(file)))
  matches <- regmatches(base, regexec("bmmc_([A-Za-z0-9]+)_vs_([A-Za-z0-9]+)", base))[[1]]
  timepoint_1 <- matches[2]
  timepoint_2 <- matches[3]

  main_title <- paste0(
    "BMMC Volcano: ", timepoint_1, " vs ", timepoint_2,
    "\n Positive log2FC = higher in ", timepoint_1,
    "; Negative log2FC = higher in ", timepoint_2
  )

  sub_title <- paste(
    "Significant genes (FDR < 0.1, |log2FC| > 0.5).",
    "Note: positive log2FC corresponds to the first group (", timepoint_1, ") in the DESeq2 contrast."
  )

  plot_volcano_by_timepoint(
    input_deg_res = file,
    title = main_title,
    subtitle = sub_title,
    timepoint_1 = timepoint_1,
    timepoint_2 = timepoint_2,
    file_save_name = base
  )
}

[volcano sanity] fraction log2FoldChange > 0 = 0.507 (timepoint_1=ASCT1y)

Warning message:
“ggrepel: 127 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 182 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 138 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 76 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 229 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
[volcano sanity] fraction log2FoldChange > 0 = 0.476 (timepoint_1=ASCT1y)

Warning message:
“ggrepel: 14 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
[volcano sanity] fraction log2FoldChange > 0 = 0.468 (timepoint_1=ASCT2y)

Warning message:
“ggrepel: 4 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
[volcano sanity] fracti

# PBMC - Longitudinal

In [7]:
plot_volcano_by_timepoint <- function(input_deg_res,
                                      title,
                                      subtitle,
                                      timepoint_1,
                                      timepoint_2,
                                      tp_color_map = c(
                                          "PreTx" = "#29216e",
                                          "PI2C" = "#0437b0",
                                          "EI" = "#006dc6",
                                          "ASCT60d" = "#00afd1",
                                          "ASCT90d" = "#19c2f0",
                                          "ASCT1y" = "#2a7a4c",
                                          "ASCT2y" = "#4FBC3B",
                                          "Healthy" = "#b52267"
                                        ),
                                      file_save_name = "volcano_plot") {
  if (!file.exists(input_deg_res)) {
    stop("Error: The file does not exist. Please check the file path.")
  }

  res <- tryCatch(
    data.table::fread(input_deg_res),
    error = function(e) {
      stop("Error: Unable to read the file. Please ensure it is a valid CSV file.")
    }
  )

  res <- res[!is.na(Qvalue) & !is.na(log2FoldChange), ]
  

  # Ensure earlier timepoint is always on the left side (negative log2FC)
  tp_order <- c(
    "PreTx",
    "EI",
    "PI2C",
    "ASCT60d", 
    "ASCT90d",
    "ASCT1y",
    "ASCT2y",
    "Healthy"
  )

  # swap removed to preserve label->LFC semantics (see commit)

  # Assign significance labels
  res$significance <- "Not Significant"
  res$significance[res$Qvalue < 0.1 & res$log2FoldChange > 0.5] <- paste("Upregulated in", timepoint_1)
  res$significance[res$Qvalue < 0.1 & res$log2FoldChange < -0.5] <- paste("Upregulated in", timepoint_2)

  res$significance <- factor(
    res$significance,
    levels = c(
      paste("Upregulated in", timepoint_2),
      paste("Upregulated in", timepoint_1),
      "Not Significant"
    )
  )
  res$significance <- droplevels(res$significance)


  # Safe color lookup with fallbacks if a timepoint is not present in tp_color_map
  color1 <- if (timepoint_1 %in% names(tp_color_map)) tp_color_map[timepoint_1] else "#000000"
  color2 <- if (timepoint_2 %in% names(tp_color_map)) tp_color_map[timepoint_2] else "#888888"
  manual_colors <- setNames(
    c(as.character(color1), as.character(color2), "gray70"),
    c(paste("Upregulated in", timepoint_1), paste("Upregulated in", timepoint_2), "Not Significant")
  )

  # Sanity check: report fraction positive LFC (non-intrusive)
  frac_pos <- mean(res$log2FoldChange > 0, na.rm = TRUE)
  message(sprintf("[volcano sanity] fraction log2FoldChange > 0 = %.3f (timepoint_1=%s)", frac_pos, timepoint_1))
  # Save to PNG
  png(paste0("../../../data/rna/pseudobulk/results/deg_volcanos/", file_save_name, "_deg_volcano.png"), width = 20, height = 28, units = "in", res = 300)
  print(
    ggplot2::ggplot(
      res,
      ggplot2::aes(
        x = log2FoldChange,
        y = -log10(pvalue),
        color = significance,
        label = gene
      )
    ) +
      ggplot2::geom_point(size = 1) +
      ggrepel::geom_text_repel(
        data = subset(res, Qvalue < 0.1),
        force = 1,
        max.overlaps = 20,
        show.legend = FALSE
      ) +
      ggplot2::geom_vline(xintercept = c(-0.5, 0.5), linetype = "dashed", color = "grey50", linewidth = 0.5) +
      ggplot2::geom_hline(yintercept = -log10(0.05), linetype = "dashed", color = "grey50", linewidth = 0.5) +
      ggplot2::facet_wrap(~celltype, ncol = 4, scales = "free") +
      ggplot2::theme_minimal() +
      ggplot2::labs(
        title = title,
        subtitle = subtitle,
        x = expression(paste("Log"[2], " Fold Change")),
        y = expression(paste("-Log"[10], " (p-value)"))
      ) +
      ggplot2::theme(
        plot.title = ggplot2::element_text(hjust = 0.5, face = "bold", size = 18, margin = ggplot2::margin(b = 10)),
        axis.title = ggplot2::element_text(size = 16, face = "bold"),
        axis.text = ggplot2::element_text(size = 12),
        legend.position = "bottom",
        legend.title = ggplot2::element_blank(),
        legend.text = ggplot2::element_text(size = 12),
        strip.text = ggplot2::element_text(size = 14, hjust = 0.05),
        panel.grid.major = ggplot2::element_line(color = "grey90", linewidth = 0.2),
        panel.grid.minor = ggplot2::element_blank(),
        panel.background = ggplot2::element_rect(fill = "white", color = NA),
        panel.border = ggplot2::element_rect(colour = "black", fill = NA, linewidth = 0.8),
        panel.spacing = grid::unit(0.5, "cm")
      ) +
      ggplot2::scale_color_manual(values = manual_colors) +
      ggplot2::guides(color = ggplot2::guide_legend(override.aes = list(size = 5)))
  )
  dev.off()
}

In [8]:
input_folder <- "../../../data/rna/pseudobulk/results/deseq2_results"
files <- list.files(input_folder, pattern = "deseq2.*pbmc_.*vs.*\\.csv$", full.names = TRUE, ignore.case = TRUE)

for (file in files) {
  base <- sub("^deseq2_results_", "l3_", tools::file_path_sans_ext(basename(file)))
  matches <- regmatches(base, regexec("pbmc_([A-Za-z0-9]+)_vs_([A-Za-z0-9]+)", base))[[1]]
  timepoint_1 <- matches[2]
  timepoint_2 <- matches[3]

  title_line1 <- paste0("PBMC Volcano: ", timepoint_1, " vs ", timepoint_2)
  title_line2 <- paste0("Positive log2FC = higher in ", timepoint_1, " ; Negative log2FC = higher in ", timepoint_2)
  main_title <- paste0(title_line1, "\n", title_line2)

  sub_title <- paste(
    "Significant genes (FDR < 0.1, |log2FC| > 0.5).",
    "Note: positive log2FC corresponds to the first group (", timepoint_1, ") in the DESeq2 contrast."
  )

  plot_volcano_by_timepoint(
    input_deg_res = file,
    title = main_title,
    subtitle = sub_title,
    timepoint_1 = timepoint_1,
    timepoint_2 = timepoint_2,
    file_save_name = base
  )
}

[volcano sanity] fraction log2FoldChange > 0 = 0.499 (timepoint_1=ASCT1y)

[volcano sanity] fraction log2FoldChange > 0 = 0.508 (timepoint_1=ASCT1y)

Warning message:
“ggrepel: 3400 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 2566 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 2940 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 1285 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 496 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 2490 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 959 unlabeled data points (too many overlaps). Consider increasing max.overlaps”
Warning message:
“ggrepel: 742 unlabeled data points (too many overlaps). Consider increa